In [4]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import pickle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Embedding, Dense, Concatenate
import copy
from numpy import unique
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Embedding
from keras.layers import concatenate
from keras.utils import plot_model
import math
from tensorflow.keras.layers import Reshape
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import LSTM
import matplotlib.pyplot as plt
#from keras.utils import plot_model
import os
import torch
from collections import Counter
import tensorflow as tf
from tensorflow.keras import optimizers
from tensorflow.keras import metrics
from tensorflow.keras import losses
from transformers import pipeline
from keras.layers import Softmax
from tensorflow.keras.utils import plot_model
from tensorflow.keras import regularizers
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import category_encoders as ce

In [ ]:
pip install tensorflow category_encoders

In [ ]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
def encode_data(df, encoding, target_column="award_contract/val_total"):

    base_n_encoder_cols = ["cpv_code", "country", "language"]
    one_hot_cols = ["type_contract", "procedure", "joint_procurement_involved", "central_purchasing", "eu_fund", "framework or dynamic purchasing", "ca_type",
                "ca_activity", "procedure"]
    numerical_cols = ["duration", "nb_tenders_received", "nb_tenders_received_sme", "ac_price", "ac_weighting", "ac_cost/ac_weighting"]
    text_cols = ["short_descr", "ac_criterion", "object_contract/title", "object_descr/title", "ac_cost/ac_criterion"]

    if encoding == "onehot":
        df_encoded = pd.get_dummies(df, columns=df.drop(columns = ["award_contract/val_total", "date_conclusion_contract"]).columns, drop_first=True, dtype=int)
        df_encoded = df_encoded.sort_values(by = ["date_conclusion_contract"], axis = 0, ascending = True)
        X_train, X_test = np.split(df_encoded.drop(columns=["award_contract/val_total", "date_conclusion_contract"]).values, [int(0.8 * len(df))])
        y_train, y_test = np.split(df_encoded["award_contract/val_total"].values, [int(0.8 * len(df))])

    elif encoding == "onehot and baseN":
        encoder = ce.BaseNEncoder(cols=base_n_encoder_cols, return_df=True, base=1)
        df = encoder.fit_transform(df)
        df = pd.get_dummies(df, columns=(one_hot_cols), drop_first=True, dtype=float)

    return df

In [ ]:
def train_validate_test_split(df, train, validate):
    target_col = "award_contract/val_total"
    sort_col = "date_conclusion_contract"
    text_cols = ["short_descr", "ac_criterion", "object_contract/title", "object_descr/title", "ac_cost/ac_criterion"]

    df = df.sort_values(by = [sort_col], axis = 0, ascending = True)
    df = df.drop(columns = ([sort_col] + text_cols))
    train_indice = int(train * len(df))
    validate_indice = int(validate * len(df)) + train_indice

    train_set = df[:train_indice-1]
    val_set = df[train_indice:validate_indice-1]
    test_set = df[validate_indice:]

    X_train = train_set.drop(columns = [target_col]).astype(float).values
    y_train = train_set[target_col].astype(float).values

    X_val = val_set.drop(columns = [target_col]).astype(float).values
    y_val = val_set[target_col].astype(float).values

    X_test = test_set.drop(columns = [target_col]).astype(float).values
    y_test = test_set[target_col].astype(float).values

    return  X_train, y_train, X_val, y_val, X_test, y_test

In [ ]:
def plot_metrics(results, save_path=None):
    plt.figure(figsize=(15, 25))

    axes = []
    for i in range(1, 16):
        ax = plt.subplot(5, 3, i)
        axes.append(ax)

    for i, subset in enumerate(results.keys()):
        model_results = results[subset]["history"].history
        for j, key in enumerate(list(model_results.keys())[:int(0.5 * len(model_results.keys()))]):
            loss_train = model_results[key]
            loss_val = model_results["val_" + key]
            epochs = range(0, len(loss_train))

            axes[i * 3 + j].plot(epochs, loss_train, "g", label="Training {}".format(key))
            axes[i * 3 + j].plot(epochs, loss_val, "b", label="Validation {}".format("val_" + key))
            axes[i * 3 + j].set_title("Training and validation of {} with metric {}".format(subset, key), fontsize = 10)
            axes[i * 3 + j].set_xlabel("Epochs")
            axes[i * 3 + j].set_ylabel("{}".format(key))
            plt.tight_layout()

    handles, labels = axes[0].get_legend_handles_labels()
    plt.legend(handles, labels, bbox_to_anchor=(1, 6), loc='upper left')

    if save_path:
        plt.savefig(save_path, format='png')
    else:
        plt.show()

In [ ]:
def filter_outliers(df, upper, lower, target_col = "award_contract/val_total"):
    """This function only works on numerical columns"""
    data_array = np.sort(df[target_col].to_numpy())
    cut_off_low_indice = math.floor(lower * len(data_array))
    cut_off_high_indice = math.floor(upper * len(data_array)) - 1
    low_amount = data_array[cut_off_low_indice]
    high_amount = data_array[cut_off_high_indice]

    outlier_indices = []

    for i in range(0, len(df)):
        if df[target_col].iloc[i] > high_amount:
            outlier_indices.append(i)
        elif df[target_col].iloc[i] < low_amount:
            outlier_indices.append(i)
        else:
            continue

In [ ]:
#load the datasets
df = pd.read_pickle("Data/train_val_test/set_1")

In [ ]:
drop_no_target_var = df.loc[pd.isna(df["award_contract/val_total"]) == True].index.to_list()
df = df.drop(labels = drop_no_target_var, axis = 0)

In [ ]:
#take variables out iteratively to get model results in steps
cpv_cols = [col for col in df.columns if "cpv_code" in col]
country_cols = [col for col in df.columns if "country" in col]
type_contract_cols = [col for col in df.columns if "type_contract" in col]
ca_type_cols = [col for col in df.columns if "ca_type" in col]
language = [col for col in df.columns if "language" in col]

leave_one_out_cols = {"cpv_cols": cpv_cols,
                      "country_cols":country_cols,
                      "type_contract_cols":type_contract_cols,
                      "ca_type_cols": ca_type_cols,
                      "language":language,
                      }

In [ ]:
results = {}

for i, feature in enumerate(leave_one_out_cols.keys()):
    df_subset = copy.deepcopy(df.drop(columns = leave_one_out_cols[feature], axis=0))

    print(i, feature)

    X_train, y_train, X_val, y_val, X_test, y_test = train_validate_test_split(df_subset, 0.6, 0.2)
    input_dimension = X_train.shape[1]

    history, mae, r2_result = create_model(input_dimension=input_dimension,
             X_train = X_train,
             y_train = y_train,
             X_val = X_val,
             y_val = y_val,
             X_test = X_test,
             y_test = y_test,
             learning_rate= 0.1,
             epochs = 10)

    results["without {}".format(feature)] = {"history": history,
                                             "mae_test": mae,
                                             "R2_test": r2_result}

In [ ]:
plot_metrics(results)

In [ ]:
def create_model(input_dimension, X_train, y_train, X_val, y_val, X_test, y_test, learning_rate = 0.001, epochs = 50):
    #let's play around a little more with the keras model
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate) #tf.keras.optimizers.experimental.Adagrad(learning_rate=0.001)
    loss_function = "mae"
    metrics = ["mae", "mse"]

    #define the layers
    input_num_cat = Input(shape=input_dimension)
    x = Dense(32, activation = "relu")(input_num_cat) #kernel_regularizer=regularizers.L1L2(l1=0.005)
    x = Dropout(rate = 0.2)(x)
    x = Dense(8, activation = "relu")(x)
    x = Dropout(rate=0.2)(x)
    x = Dense(32, activation = "relu")(x)
    regression_layer = Dense(1, activation="linear")(x)
    model_num_cat = Model(inputs = [input_num_cat],
                          outputs = regression_layer)

    model_num_cat.compile(loss=loss_function,
                          optimizer = optimizer,
                          metrics = metrics)
    model_num_cat.summary()

    history = model_num_cat.fit(x = [X_train], y=y_train,
                                  validation_data = (X_val, y_val),
                                  epochs = epochs,
                                  batch_size = 32)

    y_pred = model_num_cat.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)

    metric = tf.keras.metrics.R2Score()
    metric.update_state(y_test.reshape(-1,1), y_pred)
    r2_result = metric.result()
    r2_result.numpy()

    return history, mae, r2_result

In [ ]:
#run on the entire num_cat dataset
history, mae, r2_result = create_model(input_dimension=X_train.shape[1],
             X_train = X_train,
             y_train = y_train,
             X_val = X_val,
             y_val = y_val,
             X_test = X_test,
             y_test = y_test,
             learning_rate= 0.001,
             epochs = 100)